# QA Baseline 실험: Midm-2.0-Base-Instruct vs A.X-3.1-Light

**목적**: 두 한국어 sLLM의 QA 능력을 ROUGE-L / Token F1 기준으로 비교  
**환경**: Google Colab L4 GPU (24GB VRAM)  
**데이터**: `ai/data/qa_samples.json` (general 20건 + business 20건, 총 40건)

| 모델 | 파라미터 | 양자화 |
|------|----------|--------|
| `K-intelligence/Midm-2.0-Base-Instruct` | 11.5B | bitsandbytes NF4 4-bit |
| `skt/A.X-3.1-Light` | 7B | bitsandbytes NF4 4-bit |

In [ ]:
# 셀 1: 저장소 클론 및 의존성 설치
!git clone https://github.com/SKNETWORKS-FAMILY-AICAMP/SKN21-FINAL-3TEAM.git
%cd SKN21-FINAL-3TEAM

!pip install -q transformers bitsandbytes accelerate rouge-score

In [ ]:
# 셀 2: PYTHONPATH 설정
import sys, os
sys.path.insert(0, os.getcwd())
print("PYTHONPATH 설정 완료:", os.getcwd())

In [ ]:
# 셀 3: 데이터 확인
import json
with open('ai/data/qa_samples.json', encoding='utf-8') as f:
    data = json.load(f)
general = [d for d in data if d['domain'] == 'general']
business = [d for d in data if d['domain'] == 'business']
print(f'총 {len(data)}건: general={len(general)}, business={len(business)}')

In [ ]:
# 셀 4: GPU 확인
import torch
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 셀 5: Midm-2.0-Base-Instruct 실행
# ※ 11.5B 모델, NF4 4-bit 로드 → 약 6~7GB VRAM 사용
!python ai/experiments/run_qa_baseline.py --model midm

In [ ]:
# 셀 6: A.X-3.1-Light 실행
# ※ 7B 모델, NF4 4-bit 로드 → 약 4~5GB VRAM 사용
!python ai/experiments/run_qa_baseline.py --model ax

In [ ]:
# 셀 7: 정량 결과 출력
import json

with open('ai/experiments/results/qa_quantitative.json', encoding='utf-8') as f:
    quant = json.load(f)

print("=" * 60)
print("  QA Baseline 정량 결과")
print("=" * 60)
for s in quant:
    print(f"\n모델: {s['model']}")
    print(f"  전체 Token F1: {s['overall']['token_f1']:.4f}  (n={s['overall']['count']})")
    print(f"  전체 ROUGE-L : {s['overall']['rouge_l']:.4f}")
    print(f"  일반 Token F1: {s['general']['token_f1']:.4f}  (n={s['general']['count']})")
    print(f"  업무 Token F1: {s['business']['token_f1']:.4f}  (n={s['business']['count']})")
    print(f"  평균 추론 시간: {s['avg_infer_sec']:.2f}s/sample")
    print(f"  총 추론 시간  : {s['total_infer_sec']:.1f}s")

In [ ]:
# 셀 8: 정성 샘플 출력 (10건 × 2모델)
with open('ai/experiments/results/qa_qualitative.json', encoding='utf-8') as f:
    qual = json.load(f)

qual_samples = [q for q in qual if q['qualitative_sample']]
for entry in qual_samples:
    print(f"\n[{entry['model']}] {entry['id']}")
    print(f"  Q   : {entry['question']}")
    print(f"  Gold: {entry['gold_answer']}")
    print(f"  Pred: {entry['pred_answer']}")
    print(f"  TF1={entry['token_f1']:.3f}  RL={entry['rouge_l']:.3f}")

In [ ]:
# 셀 9: 결과 파일 로컬 다운로드
from google.colab import files
files.download('ai/experiments/results/qa_quantitative.json')
files.download('ai/experiments/results/qa_qualitative.json')